# TPWRS PRP Publications by Year: Six Affinity Bands

In [69]:
from pathlib import Path

import pandas as pd
import plotly.express as px

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
benchmark_root = repo_root / "data" / "results" / "affinity_benchmark"
mapping_path = repo_root / "data" / "processed" / "ra_grouping_ra2025.csv"
band_order = ["Low-", "Low+", "Medium-", "Medium+", "High-", "High+"]
programme_order = ["IBR", "Stability", "DER", "CROF", "System Services", "Planning"]


def load_programme_mapping():
    programme_mapping = pd.read_csv(mapping_path)[["RA2025", "Grouping"]]
    return (
        programme_mapping
        .dropna()
        .drop_duplicates()
        .rename(columns={"RA2025": "RA2025_ID", "Grouping": "PRP_Name"})
    )


def assign_programme_bands(ra_adjudication_df):
    programme_mapping = load_programme_mapping()
    question_scores = ra_adjudication_df.copy()
    question_scores["RA2025_ID"] = pd.to_numeric(question_scores["RA2025_ID"], errors="coerce")
    question_scores["Final_Affinity"] = pd.to_numeric(question_scores["Final_Affinity"], errors="coerce")
    programme_mapping["RA2025_ID"] = pd.to_numeric(programme_mapping["RA2025_ID"], errors="coerce")
    question_scores = question_scores.merge(programme_mapping, on="RA2025_ID", how="left")
    question_scores = question_scores.dropna(
        subset=["Abstract_Index", "PRP_Name", "RA2025_ID", "Final_Affinity"]
    )
    programme_scores = (
        question_scores
        .sort_values(
            ["Abstract_Index", "PRP_Name", "Final_Affinity", "RA2025_ID"],
            ascending=[True, True, False, True],
        )
        .drop_duplicates(["Abstract_Index", "PRP_Name"], keep="first")
        .rename(
            columns={
                "RA2025_ID": "Highest_Question_ID",
                "RA_Question": "Highest_Question_Text",
                "Final_Affinity": "Highest_Question_Affinity",
            }
        )
    )
    programme_scores["Affinity_Band"] = pd.cut(
        programme_scores["Highest_Question_Affinity"],
        bins=[0, 20, 40, 60, 80, 100, 120],
        labels=band_order,
        include_lowest=True,
    )
    return programme_scores


assignments_df = pd.concat(
    [
        assign_programme_bands(pd.read_csv(path)).assign(
            Year=int(path.parent.name.split("_")[1])
        )
        for path in sorted(benchmark_root.glob("tpwrs_*/adjudicated_ra_affinities.csv"))
    ],
    ignore_index=True,
 )

papers_per_year = assignments_df.groupby("Year")["Abstract_Index"].nunique()
coverage_bands = (
    assignments_df
    .groupby(["Year", "PRP_Name", "Affinity_Band"], observed=True)["Abstract_Index"]
    .nunique()
    .rename("Papers")
    .reindex(
        pd.MultiIndex.from_product(
            [papers_per_year.index, programme_order, band_order],
            names=["Year", "PRP_Name", "Affinity_Band"],
        ),
        fill_value=0,
    )
    .reset_index()
    .merge(papers_per_year.rename("Papers_Total"), on="Year")
 )
coverage_bands["Share"] = 100 * coverage_bands["Papers"] / coverage_bands["Papers_Total"]
coverage_bands["Affinity_Band"] = pd.Categorical(
    coverage_bands["Affinity_Band"], categories=band_order, ordered=True
 )


figure = px.area(
    coverage_bands,
    x="Year",
    y="Share",
    color="Affinity_Band",
    facet_col="PRP_Name",
    facet_col_wrap=2,
    category_orders={
        "PRP_Name": programme_order,
        "Affinity_Band": band_order[::-1],
    },
 )
# figure.update_yaxes(range=[0, 80])
figure.show()

In [72]:
group_highest_scores = (
    assignments_df
    .groupby(["PRP_Name", "Affinity_Band"], observed=True)["Highest_Question_Affinity"]
    .transform("max")
)

highest_scoring_indices_by_prp_band = (
    assignments_df
    .loc[assignments_df["Highest_Question_Affinity"].eq(group_highest_scores)]
    .groupby(["PRP_Name", "Affinity_Band"], observed=True)
    .agg(
        Highest_Score=("Highest_Question_Affinity", "first"),
        Abstract_Indices=("Abstract_Index", lambda indices: sorted(indices.unique())),
        Year_Abstract_Indices=(
            "Abstract_Index",
            lambda indices: list(
                zip(
                    assignments_df.loc[indices.index, "Year"],
                    indices,
                    strict=True,
                )
            ),
        ),
    )
    .reindex(
        pd.MultiIndex.from_product(
            [programme_order, band_order], names=["PRP_Name", "Affinity_Band"]
        )
    )
)
highest_scoring_indices_by_prp_band

Highest_Score  \
PRP_Name        Affinity_Band                  
IBR             Low-                    20.0   
                Low+                    40.0   
                Medium-                 60.0   
                Medium+                 80.0   
                High-                   97.6   
                High+                    NaN   
Stability       Low-                    20.0   
                Low+                    40.0   
                Medium-                 60.0   
                Medium+                 80.0   
                High-                  100.0   
                High+                  105.8   
DER             Low-                    20.0   
                Low+                    40.0   
                Medium-                 60.0   
                Medium+                 80.0   
                High-                   98.2   
                High+                  102.0   
CROF            Low-                    20.0   
                Low+                    40.0   
                Medium-                 60.0   
                Medium+                 80.0   
                High-                   99.6   
                High+                  109.6   
System Services Low-                    20.0   
                Low+                    40.0   
                Medium-                 60.0   
                Medium+                 80.0   
                High-                  100.0   
                High+                  105.6   
Planning        Low-                    20.0   
                Low+                    40.0   
                Medium-                 60.0   
                Medium+                 80.0   
                High-                   99.8   
                High+                  105.2   

                                                                Abstract_Indices  \
PRP_Name        Affinity_Band                                                      
IBR             Low-           [10.1109/TPWRS.2015.2395428, 10.1109/TPWRS.201...   
                Low+           [10.1109/TPWRS.2014.2360520, 10.1109/TPWRS.201...   
                Medium-        [10.1109/TPWRS.2022.3155630, 10.1109/TPWRS.202...   
                Medium+        [10.1109/TPWRS.2016.2608927, 10.1109/TPWRS.202...   
                High-                               [10.1109/TPWRS.2020.2968114]   
                High+                                                        NaN   
Stability       Low-           [10.1109/TPWRS.2014.2335158, 10.1109/TPWRS.201...   
                Low+           [10.1109/TPWRS.2014.2369428, 10.1109/TPWRS.201...   
                Medium-        [10.1109/TPWRS.2015.2437372, 10.1109/TPWRS.201...   
                Medium+        [10.1109/TPWRS.2014.2336859, 10.1109/TPWRS.201...   
                High-                               [10.1109/TPWRS.2026.3684993]   
                High+                               [10.1109/TPWRS.2025.3616391]   
DER             Low-           [10.1109/TPWRS.2014.2321793, 10.1109/TPWRS.201...   
                Low+           [10.1109/TPWRS.2016.2520397, 10.1109/TPWRS.201...   
                Medium-        [10.1109/TPWRS.2014.2320519, 10.1109/TPWRS.201...   
                Medium+        [10.1109/TPWRS.2018.2890169, 10.1109/TPWRS.202...   
                High-                               [10.1109/TPWRS.2022.3196708]   
                High+          [10.1109/TPWRS.2021.3052958, 10.1109/TPWRS.202...   
CROF            Low-           [10.1109/TPWRS.2014.2360520, 10.1109/TPWRS.201...   
                Low+           [10.1109/TPWRS.2014.2340131, 10.1109/TPWRS.201...   
                Medium-        [10.1109/TPWRS.2015.2479089, 10.1109/TPWRS.201...   
                Medium+                             [10.1109/TPWRS.2025.3612225]   
                High-                               [10.1109/TPWRS.2022.3211452]   
                High+                               [10.1109/TPWRS.2026.3690326]   
System Services Low-  

In [26]:
coverage_bands.groupby(["PRP_Name", "Affinity_Band"])["Papers"].sum()

/tmp/ipykernel_2427/2510727021.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  coverage_bands.groupby(["PRP_Name", "Affinity_Band"])["Papers"].sum()


PRP_Name         Affinity_Band
CROF             Low-              422
                 Low+             1941
                 Medium-          1982
                 Medium+          1320
                 High-             330
                 High+              10
DER              Low-             2773
                 Low+             1955
                 Medium-           743
                 Medium+           416
                 High-             115
                 High+               3
IBR              Low-             3279
                 Low+             1253
                 Medium-           716
                 Medium+           624
                 High-             133
                 High+               0
Planning         Low-             1905
                 Low+             1944
                 Medium-          1102
                 Medium+           789
                 High-             264
                 High+               1
Stability        Low-            

In [27]:
px.histogram(
    assignments_df,
    x= "Affinity_Band",
    # color = "Year"
    facet_col = "PRP_Name",
    facet_col_wrap=3,
    category_orders={ "Affinity_Band": band_order[::-1]}
)

In [28]:
assignments_df

,Abstract_Index,Document Title,Highest_Question_ID,Highest_Question_Text,N_Models,Affinity_Mean,Affinity_Std,Affinity_Min,Affinity_Max,Affinity_Range,...,agreement_max,Highest_Question_Affinity,Final_Reason,Final_Method,Model_Slug,Cosine_x100_Mean,Run_ID,PRP_Name,Affinity_Band,Year
0,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,27,how can risk-based methods (e.g. probabilistic...,5,31.4,14.099645,14.0,50.0,36.0,...,0.6,31.4,NaN,ensemble_mean,NaN,15.9306,tpwrs_2015_tpwrs_2015,CROF,Low+,2015
1,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,12,what mechanisms and appropriate aggregate der ...,5,15.8,12.774976,0.0,30.0,30.0,...,1.0,15.8,NaN,ensemble_mean,NaN,15.2541,tpwrs_2015_tpwrs_2015,DER,Low-,2015
2,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,1,what is the future of active power balance as ...,5,15.0,7.071068,5.0,20.0,15.0,...,1.0,15.0,NaN,ensemble_mean,NaN,33.8061,tpwrs_2015_tpwrs_2015,IBR,Low-,2015
3,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,41,what additional probabilistic planning methods...,5,53.6,22.634045,25.0,85.0,60.0,...,0.6,53.6,NaN,ensemble_mean,NaN,43.4438,tpwrs_2015_tpwrs_2015,Planning,Medium-,2015
4,10.1109/TPWRS.2014.2316271,Interval Power Flow Analysis Using Linear Rela...,9,what analytical tools and models should be pro...,5,61.4,15.485477,35.0,72.0,37.0,...,0.8,61.4,NaN,ensemble_mean,NaN,40.0692,tpwrs_2015_tpwrs_2015,Stability,Medium+,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36025,10.1109/TPWRS.2026.3707412,Frequency Quality Metrics based on Second-Orde...,15,"what aggregate quantities must be monitored, s...",5,24.8,9.757049,10.0,34.0,24.0,...,1.0,24.8,NaN,ensemble_mean,NaN,36.4899,tpwrs_2026_tpwrs_2026,DER,Low+,2026
36026,10.1109/TPWRS.2026.3707412,Frequency Quality Metrics based on Second-Orde...,1,what is the future of active power balance as ...,5,75.8,10.871982,60.0,90.0,30.0,...,0.6,75.8,NaN,ensemble_mean,NaN,39.2350,tpwrs_2026_tpwrs_2026,IBR,Medium+,2026
36027,10.1109/TPWRS.2026.3707412,Frequency Quality Metrics based on Second-Orde...,37,what studies and metrics are required to ident...,5,25.6,9.316652,18.0,40.0,22.0,...,0.8,25.6,NaN,ensemble_mean,NaN,33.3348,tpwrs_2026_tpwrs_2026,Planning,Low+,2026
36028,10.1109/TPWRS.2026.3707412,Frequency Quality Metrics based on Second-Orde...,9,what analytical tools and models should be pro...,5,75.6,6.268971,70.0,85.0,15.0,...,0.8,75.6,NaN,ensemble_mean,NaN,51.4229,tpwrs_2026_tpwrs_2026,Stability,Medium+,2026


6005

In [67]:
paper_programme_bands = assignments_df.pivot(
    index=["Year", "Abstract_Index"],
    columns="PRP_Name",
    values="Affinity_Band",
)
target_band_order = band_order

source_target_band_summaries = []
for source_prp in programme_order:
    source_prp_is_high = paper_programme_bands[source_prp].isin(["High-", "High+"])

    for target_prp in programme_order:
        if target_prp == source_prp:
            continue

        for affinity_band in target_band_order:
            target_prp_in_band = paper_programme_bands[target_prp].eq(affinity_band)
            papers_with_high_source_and_target_band = source_prp_is_high & target_prp_in_band
            conditional_probability = papers_with_high_source_and_target_band[source_prp_is_high].mean()
            target_prp_baseline_probability = target_prp_in_band.mean()
            lift = (
                conditional_probability / target_prp_baseline_probability
                if target_prp_baseline_probability > 0
                else float("nan")
            )

            source_target_band_summaries.append(
                {
                    "Source_PRP": source_prp,
                    "Target_PRP": target_prp,
                    "Affinity_Band": affinity_band,
                    "Papers_with_High_Source_PRP": source_prp_is_high.sum(),
                    "Papers_with_High_Source_and_Target_PRP_Band": (
                        papers_with_high_source_and_target_band.sum()
                    ),
                    "Probability_of_Target_PRP_Band_given_High_Source_PRP": (
                        conditional_probability
                    ),
                    "Baseline_Probability_of_Target_PRP_Band": target_prp_baseline_probability,
                    "Lift_vs_Target_PRP_Baseline": lift,
                }
            )

source_target_band_diversification_summary = pd.DataFrame(source_target_band_summaries)
source_target_band_diversification_summary

,Source_PRP,Target_PRP,Affinity_Band,Papers_with_High_Source_PRP,Papers_with_High_Source_and_Target_PRP_Band,Probability_of_Target_PRP_Band_given_High_Source_PRP,Baseline_Probability_of_Target_PRP_Band,Lift_vs_Target_PRP_Baseline
0,IBR,Stability,Low-,133,0,0.000000,0.140550,0.000000
1,IBR,Stability,Low+,133,6,0.045113,0.360200,0.125244
2,IBR,Stability,Medium-,133,28,0.210526,0.281599,0.747611
3,IBR,Stability,Medium+,133,82,0.616541,0.165196,3.732188
4,IBR,Stability,High-,133,13,0.097744,0.050958,1.918153
...,...,...,...,...,...,...,...,...
175,Planning,System Services,Low+,265,163,0.615094,0.418485,1.469814
176,Planning,System Services,Medium-,265,46,0.173585,0.165862,1.046564
177,Planning,System Services,Medium+,265,27,0.101887,0.086428,1.178864
178,Planning,System Services,High-,265,7,0.026415,0.039134,0.674990


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

source_prp_heatmaps = {}
for source_prp in programme_order:
    source_prp_heatmaps[source_prp] = (
        source_target_band_diversification_summary
        .query("Source_PRP == @source_prp")
        .pivot(
            index="Target_PRP",
            columns="Affinity_Band",
            values="Probability_of_Target_PRP_Band_given_High_Source_PRP",
        )
        .reindex(
            index=[prp for prp in programme_order if prp != source_prp],
            columns=target_band_order,
        )
    )

heatmap_figure = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[f"High source: {source_prp}" for source_prp in programme_order],
    vertical_spacing=0.1,
)

for index, source_prp in enumerate(programme_order):
    source_prp_heatmap = source_prp_heatmaps[source_prp]
    heatmap_figure.add_trace(
        go.Heatmap(
            z=source_prp_heatmap,
            x=source_prp_heatmap.columns,
            y=source_prp_heatmap.index,
            coloraxis="coloraxis",
            texttemplate="%{z:.1%}",
            textfont={"color": "#0b0b0b"},
            xgap=2,
            ygap=2,
            hovertemplate=(
                f"High source PRP: {source_prp}<br>"
                "Target PRP: %{y}<br>"
                "Affinity band: %{x}<br>"
                "Probability: %{z:.1%}<extra></extra>"
            ),
        ),
        row=index // 2 + 1,
        col=index % 2 + 1,
    )

heatmap_figure.update_layout(
    title="Target Affinity Bands Given a High-Scoring Source PRP",
    height=950,
    template="plotly_white",
    coloraxis={
        "colorscale": [
            [0, "#fcfcfb"],
            [0.5, "#86b6ef"],
            [0.75, "#2a78d6"],
            [1, "#104281"],
        ],
        "cmin": 0,
        "cmax": source_target_band_diversification_summary[
            "Probability_of_Target_PRP_Band_given_High_Source_PRP"
        ].max(),
        "colorbar": {"title": "Probability", "tickformat": ".0%"},
    },
)
heatmap_figure.update_xaxes(title_text="Target affinity band")
heatmap_figure.update_yaxes(title_text="Target PRP")
heatmap_figure.show()

In [68]:
lift_heatmap_figure = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[f"High source: {source_prp}" for source_prp in programme_order],
    vertical_spacing=0.1,
)

for index, source_prp in enumerate(programme_order):
    source_prp_lift_heatmap = (
        source_target_band_diversification_summary
        .query("Source_PRP == @source_prp")
        .pivot(
            index="Target_PRP",
            columns="Affinity_Band",
            values="Lift_vs_Target_PRP_Baseline",
        )
        .reindex(
            index=[prp for prp in programme_order if prp != source_prp],
            columns=target_band_order,
        )
    )

    lift_heatmap_figure.add_trace(
        go.Heatmap(
            z=source_prp_lift_heatmap,
            x=source_prp_lift_heatmap.columns,
            y=source_prp_lift_heatmap.index,
            coloraxis="coloraxis",
            texttemplate="%{z:.2f}x",
            textfont={"color": "#0b0b0b"},
            xgap=2,
            ygap=2,
            hovertemplate=(
                f"High source PRP: {source_prp}<br>"
                "Target PRP: %{y}<br>"
                "Affinity band: %{x}<br>"
                "Lift: %{z:.2f}x<extra></extra>"
            ),
        ),
        row=index // 2 + 1,
        col=index % 2 + 1,
    )

lift_heatmap_figure.update_layout(
    title="Lift of Target Affinity Bands Given a High-Scoring Source PRP",
    height=950,
    template="plotly_white",
    coloraxis={
        "colorscale": [
            [0, "#e34948"],
            [0.5, "#fcfcfb"],
            [1, "#2a78d6"],
        ],
        "cmin": 0,
        "cmid": 1,
        "colorbar": {"title": "Lift vs. baseline", "tickformat": ".1f"},
    },
)
lift_heatmap_figure.update_xaxes(title_text="Target affinity band")
lift_heatmap_figure.update_yaxes(title_text="Target PRP")
lift_heatmap_figure.show()